# 02 Baselines

The simplest possible recommenders. Every real model (KNN, SVD, content-based, hybrid) must beat these, otherwise it is not learning anything useful.

| Model | Predicts a rating? | Ranks movies? |
|---|---|---|
| Global mean | yes: the same average for everything | no |
| Movie mean | yes: each movie's average | yes: highest average first |
| Movie mean (damping=10) | yes: average pulled toward the global mean when a movie has few ratings | yes |
| Popularity | no | yes: most-rated first |

Metrics come from `src/metrics.py`: RMSE/MAE (lower is better) and Precision/Recall/NDCG@10 (higher is better; a test movie is relevant if rated >= 4.0).

In [1]:
import sys
sys.path.insert(0, "..")  # so that `src` can be imported from the notebooks folder

import numpy as np
import pandas as pd

from src.baselines import GlobalMean, MovieMean, Popularity
from src.data import load_ratings, random_split, user_time_split
from src.metrics import evaluate_ranking, mae, rmse

K = 10
ratings = load_ratings()

In [2]:
def evaluate(model, train, test, k=K):
    """Fit the model on train and return one row of the results table."""
    model.fit(train)
    predictions = model.predict(test)
    has_ratings = not np.isnan(predictions).all()
    users = test["userId"].unique()
    recommendations = {user: model.recommend(user, k) for user in users}
    has_ranking = any(recommendations.values())
    ranking = evaluate_ranking(recommendations, test, k)
    return {
        "model": model.name,
        "rmse": rmse(test["rating"], predictions) if has_ratings else np.nan,
        "mae": mae(test["rating"], predictions) if has_ratings else np.nan,
        f"precision@{k}": ranking[f"precision@{k}"] if has_ranking else np.nan,
        f"recall@{k}": ranking[f"recall@{k}"] if has_ranking else np.nan,
        f"ndcg@{k}": ranking[f"ndcg@{k}"] if has_ranking else np.nan,
    }

In [3]:
splits = {"user_time": user_time_split, "random": random_split}
models = [GlobalMean(), MovieMean(), MovieMean(damping=10), Popularity()]

rows = []
for split_name, split in splits.items():
    train, test = split(ratings)
    for model in models:
        rows.append({"split": split_name} | evaluate(model, train, test))

results = pd.DataFrame(rows)
results.round(4)

,split,model,rmse,mae,precision@10,recall@10,ndcg@10
0,user_time,Global mean,1.0688,0.8360,NaN,NaN,NaN
1,user_time,Movie mean,1.0227,0.7867,0.0000,0.0000,0.0000
2,user_time,Movie mean (damping=10),1.0051,0.7777,0.0428,0.0330,0.0551
3,user_time,Popularity,NaN,NaN,0.0563,0.0507,0.0741
4,random,Global mean,1.0488,0.8316,NaN,NaN,NaN
5,random,Movie mean,0.9827,0.7604,0.0002,0.0001,0.0002
6,random,Movie mean (damping=10),0.9735,0.7568,0.0740,0.0608,0.0959
7,random,Popularity,NaN,NaN,0.1225,0.1028,0.1587


## Findings

- **Movie mean beats global mean on RMSE/MAE.** Knowing which movie it is helps: some movies are simply rated higher than others.
- **Plain movie mean is useless for ranking** (precision@10 is about 0). The top of its list is movies with a single 5.0 rating that nobody else has seen. Damping fixes this: with `damping=10`, a movie needs many good ratings to rank high.
- **Popularity is the strongest ranking baseline.** Popular movies are what most people watch next. CF and content-based models must beat it.
- **The `user_time` split is harder than `random`** for every model, because it asks the model to predict each user's *future* ratings. The final experiments use `user_time` (see `docs/PROJECT_OVERVIEW.md`).

In [4]:
results.to_csv("../docs/results.csv", index=False)
print("Saved to docs/results.csv")

Saved to docs/results.csv
